# 11.1.2 Chroma Embedding 데이터 저장 테스트

이 노트북은 `sentence-transformers/all-MiniLM-L6-v2` 모델로 문장을 임베딩하고, Chroma DB에 저장한 뒤 유사도 검색을 수행하는 예제입니다.

원본 txt 예제의 핵심 흐름:

1. 샘플 부동산 설명 문장 준비
2. SentenceTransformer로 임베딩 생성
3. Chroma PersistentClient로 컬렉션 생성
4. 문장과 임베딩 저장
5. 질의 문장을 임베딩하여 유사한 문장 3개 검색

## 1. 패키지 확인

이미 프로젝트 가상환경에 `chromadb`, `sentence-transformers`가 설치되어 있으면 별도 설치 없이 실행합니다.

필요할 때만 아래 명령을 터미널에서 실행하세요.

```powershell
.\.venv\Scripts\python.exe -m pip install chromadb sentence-transformers
```

In [1]:
import warnings
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.tokenization_utils_base")

print("chromadb:", chromadb.__version__)

chromadb: 1.5.9


## 2. 샘플 데이터 준비

In [2]:
sentences = [
    "A stylish 3 bedroom townhouse close to the river with a private balcony.",
    "A serene 4 bedroom cabin in the mountains with a cozy fireplace.",
    "A sleek studio apartment in a vibrant neighborhood with a shared gym.",
    "A charming 2 bedroom cottage in the countryside with scenic views.",
    "A bright 2 bedroom flat near the university with a spacious living area.",
    "A modern 1 bedroom loft in the heart of the city with a rooftop terrace.",
]

for idx, sentence in enumerate(sentences):
    print(f"{idx}: {sentence}")

0: A stylish 3 bedroom townhouse close to the river with a private balcony.
1: A serene 4 bedroom cabin in the mountains with a cozy fireplace.
2: A sleek studio apartment in a vibrant neighborhood with a shared gym.
3: A charming 2 bedroom cottage in the countryside with scenic views.
4: A bright 2 bedroom flat near the university with a spacious living area.
5: A modern 1 bedroom loft in the heart of the city with a rooftop terrace.


## 3. 임베딩 생성

처음 실행할 때는 Hugging Face 모델 다운로드 때문에 시간이 걸릴 수 있습니다.

In [3]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

embeddings = model.encode(sentences)

print("embedding shape:", embeddings.shape)
print("first embedding length:", len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding shape: (6, 384)
first embedding length: 384


## 4. Chroma DB 컬렉션 생성 및 저장

노트북을 반복 실행해도 테스트 결과가 꼬이지 않도록 기존 컬렉션이 있으면 삭제 후 다시 생성합니다.

Chroma 데이터는 `04_vectordb/chroma_store` 폴더에 저장됩니다.

In [4]:
persist_dir = Path("chroma_store")
collection_name = "Apt103"

client = chromadb.PersistentClient(path=str(persist_dir))

try:
    client.delete_collection(collection_name)
except Exception:
    pass

collection = client.get_or_create_collection(collection_name)

collection.add(
    ids=[str(idx) for idx in range(len(sentences))],
    embeddings=[embedding.tolist() for embedding in embeddings],
    metadatas=[{"description": sentence} for sentence in sentences],
)

print("collection:", collection_name)
print("count:", collection.count())

collection: Apt103
count: 6


## 5. 유사도 검색

In [5]:
query = "bright house near a university"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=["metadatas", "distances"],
)

print("Query:", query)
print()

for idx, (metadata, distance) in enumerate(zip(results["metadatas"][0], results["distances"][0]), start=1):
    print(f"Result {idx}: {metadata['description']} (Distance: {distance})")

Query: bright house near a university

Result 1: A bright 2 bedroom flat near the university with a spacious living area. (Distance: 0.5602437257766724)
Result 2: A sleek studio apartment in a vibrant neighborhood with a shared gym. (Distance: 1.0175741910934448)
Result 3: A modern 1 bedroom loft in the heart of the city with a rooftop terrace. (Distance: 1.1594297885894775)


## 6. 결과 해석

`Distance` 값은 검색 질의와 저장된 문장 사이의 거리입니다.

- 값이 작을수록 더 유사한 문장입니다.
- 예제에서는 `bright`, `near`, `university`와 의미가 가까운 문장이 상위에 나와야 합니다.
- 컬렉션 이름을 바꾸면 여러 실습 결과를 분리해서 저장할 수 있습니다.